# GANForge -- cGAN (Standalone Notebook)

**Class-conditional generation**

This is a **self-contained extract** of the cGAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="2"></a>
## 2. cGAN -- Class-conditional generation

### 2.1 Problem Statement & Why this GAN

Mirza & Osindero (2014) fixed the "you cannot ask for a 7" limitation of
Section 1 by feeding a **label `y`** into *both* `G` and `D`. The generator
now models `p(x | y)` -- generate a *specific requested class* -- and the
discriminator must judge realism *conditioned on that same label*, so it
can no longer be fooled by a realistic image of the wrong class. This is
the **class-conditional generation** problem: user picks the category, the
model draws it.

We train on FashionMNIST (10 clothing categories) so conditioning is
visually obvious (a "sneaker" looks nothing like a "coat").

### 2.2 Data Collection & Preparation

In [ ]:
CGAN_IMG_SIZE = 32
CGAN_BATCH_SIZE = 128
CGAN_NZ = 100
CGAN_N_CLASSES = 10
CGAN_EPOCHS = 5
CGAN_CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                     "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

cgan_transform = T.Compose([T.Resize(CGAN_IMG_SIZE), T.ToTensor(), T.Normalize((0.5,), (0.5,))])

try:
    cgan_dataset = torchvision.datasets.FashionMNIST(root="data", train=True, download=True, transform=cgan_transform)
    print(f"Loaded real FashionMNIST: {len(cgan_dataset)} images")
except Exception as e:
    print(f"FashionMNIST download failed ({e}); falling back to synthetic per-class shapes.")
    rng = np.random.default_rng(SEED)
    n_per_class = 200
    imgs, labels = [], []
    yy, xx = torch.meshgrid(torch.arange(CGAN_IMG_SIZE), torch.arange(CGAN_IMG_SIZE), indexing="ij")
    for c in range(CGAN_N_CLASSES):
        for _ in range(n_per_class):
            cx, cy = rng.uniform(10, 22, 2)
            r = 3 + c  # radius encodes class -> a learnable, class-distinguishing synthetic signal
            blob = ((xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2).float()
            imgs.append(blob * 2 - 1)
            labels.append(c)
    cgan_dataset = TensorDataset(torch.stack(imgs).unsqueeze(1), torch.tensor(labels))

cgan_loader = DataLoader(cgan_dataset, batch_size=CGAN_BATCH_SIZE, shuffle=True, drop_last=True)
real_batch, real_labels = next(iter(cgan_loader))
show_images(real_batch[:64], title="Real conditional dataset samples", nrow=8)


### 2.3 Model Architecture

In [ ]:
class CGAN_Generator(nn.Module):
    '''Same DCGAN backbone as Section 1, but the class label is embedded
    into a vector and concatenated with the noise vector before the first
    transposed conv. This is the simplest and most common way to condition
    a generator: give it the extra information as extra input channels.'''
    def __init__(self, nz=CGAN_NZ, n_classes=CGAN_N_CLASSES, ngf=64, nc=1, embed_dim=50):
        super().__init__()
        self.label_embed = nn.Embedding(n_classes, embed_dim)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz + embed_dim, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z, y):
        y_emb = self.label_embed(y)                      # (B, embed_dim)
        zy = torch.cat([z, y_emb], dim=1)                 # (B, nz+embed_dim)
        return self.net(zy.view(zy.size(0), -1, 1, 1))


class CGAN_Discriminator(nn.Module):
    '''The label is embedded and projected to a full HxW "label map" that is
    concatenated as an extra image channel -- lets every conv layer see
    "what class am I judging" from the very first layer.'''
    def __init__(self, n_classes=CGAN_N_CLASSES, nc=1, ndf=64, img_size=CGAN_IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        self.label_embed = nn.Embedding(n_classes, img_size * img_size)
        self.net = nn.Sequential(
            nn.Conv2d(nc + 1, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x, y):
        y_map = self.label_embed(y).view(-1, 1, self.img_size, self.img_size)
        xy = torch.cat([x, y_map], dim=1)
        return self.net(xy).view(-1)


nc_cgan = real_batch.shape[1]
G_cgan = CGAN_Generator(nc=nc_cgan).to(DEVICE).apply(weights_init)
D_cgan = CGAN_Discriminator(nc=nc_cgan).to(DEVICE).apply(weights_init)


### 2.4 Loss Functions & Optimizers

In [ ]:
cgan_criterion = nn.BCEWithLogitsLoss()
cgan_opt_G = torch.optim.Adam(G_cgan.parameters(), lr=2e-4, betas=(0.5, 0.999))
cgan_opt_D = torch.optim.Adam(D_cgan.parameters(), lr=2e-4, betas=(0.5, 0.999))
cgan_fixed_noise = torch.randn(CGAN_N_CLASSES * 8, CGAN_NZ, device=DEVICE)
cgan_fixed_labels = torch.arange(CGAN_N_CLASSES, device=DEVICE).repeat_interleave(8)  # 8 samples per class, ordered -> easy visual check


### 2.5 Training Loop

In [ ]:
cgan_history = []
for epoch in range(CGAN_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter()}
    pbar = tqdm(cgan_loader, desc=f"[cGAN] epoch {epoch+1}/{CGAN_EPOCHS}")
    for real, y in pbar:
        real, y = real.to(DEVICE), y.to(DEVICE)
        b = real.size(0)

        D_cgan.zero_grad(set_to_none=True)
        logits_real = D_cgan(real, y)
        loss_d_real = cgan_criterion(logits_real, torch.ones(b, device=DEVICE))
        z = torch.randn(b, CGAN_NZ, device=DEVICE)
        fake_y = torch.randint(0, CGAN_N_CLASSES, (b,), device=DEVICE)  # ask for RANDOM classes, not the real batch's classes
        fake = G_cgan(z, fake_y)
        logits_fake = D_cgan(fake.detach(), fake_y)
        loss_d_fake = cgan_criterion(logits_fake, torch.zeros(b, device=DEVICE))
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        cgan_opt_D.step()

        G_cgan.zero_grad(set_to_none=True)
        logits_fake_for_g = D_cgan(fake, fake_y)
        loss_g = cgan_criterion(logits_fake_for_g, torch.ones(b, device=DEVICE))
        loss_g.backward()
        cgan_opt_G.step()

        meters["D"].update(loss_d.item()); meters["G"].update(loss_g.item())
        pbar.set_postfix(loss_D=f"{loss_d.item():.3f}", loss_G=f"{loss_g.item():.3f}")

    cgan_history.append((meters["G"].avg, meters["D"].avg))
    with torch.no_grad():
        G_cgan.eval()
        sample = G_cgan(cgan_fixed_noise, cgan_fixed_labels)
        G_cgan.train()
    show_images(sample, title=f"cGAN samples after epoch {epoch+1} (rows = classes)", nrow=8)
    save_checkpoint({"G": G_cgan.state_dict(), "D": D_cgan.state_dict()}, f"{CHECKPOINT_DIR}/cgan_epoch{epoch+1}.pt")

print("cGAN training complete.")


### 2.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def cgan_generate(class_id: int, n_samples: int = 8, seed: int = None) -> torch.Tensor:
    '''Generate n_samples of a user-specified class.'''
    if seed is not None:
        torch.manual_seed(seed)
    G_cgan.eval()
    z = torch.randn(n_samples, CGAN_NZ, device=DEVICE)
    y = torch.full((n_samples,), class_id, dtype=torch.long, device=DEVICE)
    imgs = G_cgan(z, y)
    G_cgan.train()
    return imgs

show_images(cgan_generate(class_id=7, n_samples=8), title=f"cGAN inference: class='{CGAN_CLASS_NAMES[7]}'", nrow=8, figsize=(10, 2))


### 2.7 Visualization & Results

In [ ]:
gs, ds = zip(*cgan_history)
plt.figure(figsize=(6, 4)); plt.plot(gs, label="G"); plt.plot(ds, label="D")
plt.legend(); plt.title("cGAN training losses"); plt.xlabel("epoch"); plt.show()

# All classes side by side -- the core deliverable of conditional generation.
with torch.no_grad():
    G_cgan.eval()
    z = torch.randn(CGAN_N_CLASSES * 6, CGAN_NZ, device=DEVICE)
    y = torch.arange(CGAN_N_CLASSES, device=DEVICE).repeat_interleave(6)
    grid = G_cgan(z, y)
    G_cgan.train()
show_images(grid, title="One row per class: " + ", ".join(CGAN_CLASS_NAMES), nrow=6, figsize=(6, 12))


### 2.8 User Input → Output Demo

In [ ]:
# Change this to any class name from CGAN_CLASS_NAMES and re-run.
USER_REQUESTED_CLASS = "Sneaker"
class_idx = CGAN_CLASS_NAMES.index(USER_REQUESTED_CLASS)
show_images(cgan_generate(class_idx, n_samples=16), title=f"User requested: '{USER_REQUESTED_CLASS}'", nrow=4)


### 2.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Conditioning information can be injected almost anywhere
  (input concatenation, as here; label-conditional BatchNorm as in BigGAN;
  projection discriminator; cross-attention as in modern text-to-image
  models) -- the *choice of injection point* is one of the biggest levers
  for conditional-GAN quality.
- ⚠️ **Warning.** With a small/simple discriminator, cGAN can learn to
  mostly ignore the label and rely on the discriminator being lenient
  (**label leakage/ignoring**) -- always sanity-check that requesting
  different classes actually produces visibly different outputs (Section 2.7).

**This GAN solved the problem of: class-conditional generation --
letting the user choose *which* category to generate, not just generating
an arbitrary sample from the whole dataset.**